# Clustering and segmentation

*Feature scaling, k-means, the choice of k, and segment profiles*

With this chapter we begin Part V, the last stage of the course map in [Chapter 1](https://pyba.murtaza.cc/parts/part-01-foundations/ch-01-analytics-and-python.html): deciding what to do. So far in this course, every model that we have built had a label to learn from, for example, whether a customer churned or stayed ([Chapters 9](https://pyba.murtaza.cc/parts/part-04-predictive/ch-09-classification-1.html) and [10](https://pyba.murtaza.cc/parts/part-04-predictive/ch-10-classification-2.html)), a customer's revenue over 12 months ([Chapter 8](https://pyba.murtaza.cc/parts/part-03-statistical/ch-08-regression.html)), or this week's revenue ([Chapter 11](https://pyba.murtaza.cc/parts/part-04-predictive/ch-11-forecasting.html)). However, some questions are not answered by labels or target values. One such question at Prairie Wholesale is how to treat different customers: which customers should receive a named account manager, which should get a retention offer, and which should receive free shipping. With 872 customers, such choices are made for groups of similar customers. The marketing team therefore asks what types of customers the company has. This question cannot be answered by any of the existing columns in the customer data. We can approach such a problem with a technique called **clustering**, in which data points (here, customers) are placed in groups on the basis of their features alone, so that the points in the same group are similar to each other and different from the points in other groups. In marketing, clusters of customers are called **segments**, and the grouping of customers into segments is called **segmentation**. In this chapter we segment the customers with k-means, choose the number of segments, describe each segment in the original units, and add the segments to the Chapter 6 dashboard.

The figure shows the result of a clustering with hypothetical data. Before the clustering, every customer looks the same, as a point described by its features. After the clustering, each customer is assigned to the cluster whose typical values are closest to its own, and each cluster is named from the typical values of its customers.

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
import numpy as _np
import plotly.graph_objects as _go
from plotly.subplots import make_subplots as _subplots

# hypothetical customers: orders a year and average order value for three types of buyer
_rng = _np.random.default_rng(12)
_spec = [("Large, infrequent orders", 7, 2.0, 1900, 0.25, "#2b6cb0"),
         ("Frequent buyers", 32, 5.0, 430, 0.22, "#bf8700"),
         ("Small orders", 11, 3.0, 85, 0.3, "#8250df")]
_pts = [( _rng.normal(mo, so, 40).clip(1), _np.exp(_rng.normal(_np.log(mv), sv, 40)), name, col)
        for name, mo, so, mv, sv, col in _spec]
_f = _subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04,
               subplot_titles=["<b>Before:</b> customers on two features", "<b>After:</b> three clusters"])
for _x, _y, _name, _col in _pts:
    _f.add_trace(_go.Scatter(x=_x, y=_y, mode="markers", marker=dict(size=6, color="#9aa1a9", opacity=0.7),
                             showlegend=False, hoverinfo="skip"), row=1, col=1)
    _f.add_trace(_go.Scatter(x=_x, y=_y, mode="markers", marker=dict(size=6, color=_col, opacity=0.75),
                             showlegend=False, hoverinfo="skip"), row=1, col=2)
    _f.add_annotation(x=float(_np.median(_x)), y=float(_np.log10(_y.max()) + 0.12), xref="x2", yref="y2",
                      text=f"<b>{_name}</b>", showarrow=False, font=dict(size=12, color=_col))
_ticks = [50, 100, 200, 500, 1000, 2000, 5000]
_f.update_yaxes(type="log", tickvals=_ticks, ticktext=[f"${t:,}" for t in _ticks])
_f.update_yaxes(title_text="Average order value", row=1, col=1)
_f.update_xaxes(title_text="Orders a year", range=[0, 45])
_f.update_layout(height=340, margin=dict(t=40, l=10, r=10, b=10))
_f

You can run this chapter's notebook in two ways. In the cloud, [**open ch-12-clustering.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-12-clustering.ipynb); nothing needs to be installed. Locally, use the `pyba-core` environment (Appendix A). Either way, run the setup cell below first. No keys or paid accounts are needed anywhere in this book; Colab requires only a free Google account.


In [ ]:
# Setup. Run this cell once per session. It installs this chapter's
# packages; on Google Colab it also fetches the course data.
%pip install -q pandas plotly scikit-learn
import sys
if "google.colab" in sys.modules:
    !test -d pyba-companion || git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

## Supervised and unsupervised learning
In [Chapter 9](https://pyba.murtaza.cc/parts/part-04-predictive/ch-09-classification-1.html) and [Chapter 10](https://pyba.murtaza.cc/parts/part-04-predictive/ch-10-classification-2.html), every customer had a label, `churned`. The models learned from the features (input variables) and the label together. We scored each model by comparing its predictions with the labels of the test customers, held out from training as in [Chapter 9](https://pyba.murtaza.cc/parts/part-04-predictive/ch-09-classification-1.html). This is called **supervised learning**, because the models learn from features with a known label. Clustering is a form of **unsupervised learning**, in which there is no label and we look for structure in the features alone. The following table lists the differences between the two settings:

|  | Supervised (Chapters 9 and 10) | Unsupervised (this chapter) |
|---|---|---|
| Data | features and a label, `churned` | features only |
| Result | a predicted label for each customer | a group number for each customer |
| Evaluation | accuracy and error cost on held-out test customers | separation of the groups, stability across refits, and usefulness for a decision |
| Baseline | the naive model, which predicts "stay" for everyone; offers to no one or to everyone | an existing grouping, by business type |

Without a label, there is no test-set accuracy and no confusion matrix. Whether a grouping is correct cannot be read from any single number. In the evaluation section (Chapter 12), we apply the three checks in the Evaluation row above and compare the result with the business-type grouping.

We cluster the customers on their buying behavior. We use six columns of the customer snapshot, the table from Chapters 9 and 10 with one row per customer, each a measure of how a customer buys:

| Column | Meaning |
|---|---|
| `orders_12m` | number of orders in the 12 months before the cutoff |
| `avg_order_value` | average order total, in dollars, over the customer's history to the cutoff |
| `pct_online` | share of the customer's orders placed online |
| `avg_discount_pct` | average discount rate over the customer's history |
| `aug_jan_share` | share of the customer's orders placed in August or January |
| `weeks_since_last_order` | weeks between the last order and the cutoff |



In [ ]:
import pandas as pd
import plotly.express as px

from pyba import DATA_DIR

snap = pd.read_csv(DATA_DIR / "pw_customer_snapshot.csv")

features = ["orders_12m", "avg_order_value", "pct_online",
            "avg_discount_pct", "aug_jan_share", "weeks_since_last_order"]

# --- Look: the six behavior columns for the first three customers ---
snap[["customer_id"] + features].head(3)

With `+`, two lists are joined into one. `["customer_id"] + features` is therefore a list of seven column names. The first customer, C0001, placed 19 orders in the year before the cutoff and last ordered 0.9 weeks before it. The second customer, C0002, had not ordered for 21.9 weeks.

## Feature scaling

The six columns are measured in different units and on different scales:

In [ ]:
snap[features].describe().round(2).loc[["mean", "std"]]

The standard deviation of the average order value is \$709.74. The standard deviations of the three columns measured as proportions are between 0.01 and 0.28. In k-means, customers are grouped by distance: the Euclidean (straight-line) distance across all features, the distance metric used for KNN in Chapter 9. For two customers with values $p_1, \dots, p_6$ and $q_1, \dots, q_6$ on the six features, the distance is

$$\text{distance} = \sqrt{(p_1 - q_1)^2 + (p_2 - q_2)^2 + \dots + (p_6 - q_6)^2}$$

Without scaling, a difference of \$1 in average order value would count as much as the difference between a customer with no online orders and a customer with only online orders. Distances would then depend almost entirely on average order value.

In Chapter 9, the test accuracy of KNN was 77.1% without scaling, below the majority baseline of 78%, and 83.2% with scaling. In clustering, there is no test accuracy, so the same problem would go unnoticed: if the distances depend on one feature only, so do the groups.

The online edition has an interactive explorer here: k-means on orders and average order value, with a switch for scaling. Without scaling, the groups are horizontal bands of average order value, each spanning most of the range of orders. [Open the interactive explorer](https://pyba.murtaza.cc/assets/demos/kmeans-scaling.html).


Before clustering, we scale the six features, as we did for KNN in [Chapter 9](https://pyba.murtaza.cc/parts/part-04-predictive/ch-09-classification-1.html):

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled = scaler.fit_transform(snap[features])   # a NumPy array: 872 rows, 6 columns

# --- Look: the scaled values for the first three customers ---
scaled[:3].round(2)

With a `StandardScaler`, every feature is rescaled to mean 0 and standard deviation 1, as in Chapter 9: the feature's mean is subtracted, and the result is divided by its standard deviation:

$$\text{scaled value} = \frac{\text{value} - \text{mean}}{\text{standard deviation}}$$

In Chapter 9 the scaler was a step of a pipeline, and the scaler was fitted whenever the pipeline was fitted. Here we use the scaler on its own, outside a pipeline. With `fit_transform`, two steps are done in one call. The scaler is fitted, which means that each feature's mean and standard deviation are computed. Then the scaled values are returned. Because clustering has no test set, the scaler is fitted on all 872 customers.

The result is a NumPy array with one row per customer, in the order of `snap`, and one column per feature. In the second row, C0002's last order was 21.9 weeks before the cutoff. The mean is 6.13 weeks and the standard deviation is 6.73 weeks. The scaled value is therefore

$$\frac{21.9 - 6.13}{6.73} = 2.34$$

C0002's recency is 2.34 standard deviations above the mean. To keep the column names, we put the array in a DataFrame:

In [ ]:
X = pd.DataFrame(scaled, columns=features)    # the same numbers, with column names

# --- Look: every scaled column has mean 0 and standard deviation 1 ---
X.describe().round(2).loc[["mean", "std"]]

In the output, −0.0 is a small negative number rounded to zero. With the features scaled, we can now cluster the customers on `X`.

## k-means

In **k-means**, the customers are divided into k groups. The number of groups, k, is chosen before fitting. Each group has a **center**, a point with one value per scaled feature. Each customer belongs to the group with the nearest center. The centers are found with the procedure in the following table. After k starting centers are chosen, two steps, called *assign* and *update*, are repeated until no customer changes group.

| Step | What is done |
|---|---|
| Start | k starting centers are chosen, for example k customers drawn at random. |
| Assign | Each customer is assigned to the nearest center, by straight-line distance across the six scaled features. |
| Update | Each center is moved to the mean of the customers assigned to it. |
| Repeat | The assign and update steps are repeated until no customer changes group. |

The figure shows the four steps on 15 hypothetical points with k = 3, starting from a poor choice of centers.

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 15 hand-placed points in three groups (upper left, upper right, bottom), two scaled features
_pts = np.array([(-1.6, 1.2), (-1.2, 1.6), (-1.8, 0.7), (-1.0, 0.9), (-1.4, 1.0),
                 (1.2, 1.3), (1.6, 0.9), (1.0, 0.8), (1.5, 1.6), (1.9, 1.2),
                 (0.0, -1.3), (0.4, -1.0), (-0.4, -1.6), (0.3, -1.7), (-0.2, -0.9)])
_colors = ["#2b6cb0", "#bf8700", "#8250df"]

def _assign(centers):                     # index of the nearest center for every point
    return np.argmin(((_pts[:, None, :] - centers[None]) ** 2).sum(axis=2), axis=1)

def _inertia(groups, centers):
    return ((_pts - centers[groups]) ** 2).sum()

_c0 = _pts[[0, 4, 12]]                    # poor start: two centers in the upper-left group
_g1 = _assign(_c0)
_c1 = np.array([_pts[_g1 == j].mean(axis=0) for j in range(3)])
_g2 = _assign(_c1)
_c2 = np.array([_pts[_g2 == j].mean(axis=0) for j in range(3)])

_panels = [("1. Start", None, _c0), (f"2. Assign: inertia {_inertia(_g1, _c0):.1f}", _g1, _c0),
           (f"3. Update: inertia {_inertia(_g1, _c1):.1f}", _g1, _c1),
           (f"4. Assign again: inertia {_inertia(_g2, _c1):.1f}", _g2, _c1)]
_fig = make_subplots(rows=2, cols=2, shared_xaxes=True, shared_yaxes=True,
                     horizontal_spacing=0.05, vertical_spacing=0.12,
                     subplot_titles=[p[0] for p in _panels])
for _k, (_title, _groups, _centers) in enumerate(_panels):
    _r, _c = _k // 2 + 1, _k % 2 + 1
    if _groups is not None and _k != 2:   # dashed line from each point to its center
        for _p, _g in zip(_pts, _groups):
            _fig.add_trace(go.Scatter(x=[_p[0], _centers[_g][0]], y=[_p[1], _centers[_g][1]], mode="lines",
                                      line=dict(color=_colors[_g], width=1, dash="dot"),
                                      hoverinfo="skip", showlegend=False), row=_r, col=_c)
    _point_colors = ["#9aa1a9"] * len(_pts) if _groups is None else [_colors[g] for g in _groups]
    _fig.add_trace(go.Scatter(x=_pts[:, 0], y=_pts[:, 1], mode="markers", showlegend=False,
                              marker=dict(size=8, color=_point_colors, line=dict(width=1, color="white"))),
                   row=_r, col=_c)
    if _k == 2:                           # arrows from the old centers to the new ones
        for _j in range(3):
            _fig.add_annotation(x=_c1[_j][0], y=_c1[_j][1], ax=_c0[_j][0], ay=_c0[_j][1],
                                xref="x3", yref="y3", axref="x3", ayref="y3", text="", showarrow=True,
                                arrowhead=2, arrowwidth=1.5, arrowcolor=_colors[_j])
        _fig.add_trace(go.Scatter(x=_c0[:, 0], y=_c0[:, 1], mode="markers", showlegend=False, hoverinfo="skip",
                                  marker=dict(symbol="x-thin", size=12, line=dict(width=2, color="#9aa1a9"))),
                       row=_r, col=_c)
    _fig.add_trace(go.Scatter(x=_centers[:, 0], y=_centers[:, 1], mode="markers", showlegend=False,
                              marker=dict(symbol="x", size=14, color=["#333"] * 3 if _groups is None else _colors,
                                          line=dict(width=1.5, color="white"))),
                   row=_r, col=_c)
for _ann in _fig.layout.annotations[:4]:
    _ann.font.size = 12
_fig.update_xaxes(range=[-2.3, 2.3], showticklabels=False)
_fig.update_yaxes(range=[-2.1, 2.1], showticklabels=False)
_fig.update_layout(height=460, margin=dict(t=40, l=10, r=10, b=10))
_fig

The **inertia** is the sum, over all customers, of the squared distance from the customer to the center of its group:

$$\text{inertia} = \sum_{\text{customers}} (\text{distance to the center of its group})^2$$

A low inertia means that customers are close to their centers. While the assign and update steps are applied, the inertia cannot rise, because each step can only shorten the distances: in the assign step, a customer moves only to a nearer center, and in the update step, each center moves to the point with the smallest total squared distance to its customers. The steps are repeated until the inertia no longer falls. The grouping reached in this way depends on the starting centers. A different start may give us a different grouping, sometimes with a lower inertia.

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=4, n_init=10, random_state=0)
snap["segment"] = km.fit_predict(X)

# --- Look: the number of customers in each segment ---
snap["segment"].value_counts().sort_index()

`KMeans`, from scikit-learn's `cluster` module, is scikit-learn's k-means model. With `n_clusters=4`, four groups are formed. We start with four and examine the choice of k in the next section. With `fit_predict`, the model is fitted and the group number of every customer is returned, in the order of the rows of `X`. We store these numbers, the assigned segment numbers, in a new column, `segment`. With `.sort_index()`, the counts are listed in order of segment number. The four segments have 103, 353, 205, and 211 customers. The numbers 0 to 3 are labels only, with no order among the segments.

### Fitted centers

After fitting, the four centers are stored in the model. We put them in a DataFrame with the feature names:

In [ ]:
centers = pd.DataFrame(km.cluster_centers_, columns=features)

# --- Look: the four centers, in scaled units ---
centers.round(2)

The attribute `cluster_centers_` holds the fitted centers as a NumPy array, one row per segment. Each value is measured in standard deviations from the feature's mean. The center of segment 0 has an average order value 2.39 standard deviations above the mean, and an `aug_jan_share` 1.55 above it. The center of segment 2 has 1.49 standard deviations more orders than the mean.

At the end of the fitting, every customer is assigned to its nearest center. We check this for C0002, the second row of `X`:

In [ ]:
customer = X.iloc[1]                  # C0002, in scaled units

rows = []
for j in range(4):
    sq_dist = ((customer - centers.iloc[j]) ** 2).sum()    # squared distance to center j
    rows.append([j, sq_dist])

# --- Look: C0002's squared distance to each of the four centers ---
pd.DataFrame(rows, columns=["center", "squared distance"]).round(2)

With `customer - centers.iloc[j]`, the center's values are subtracted from the customer's, feature by feature, matched by column name. The six differences are squared and added up. The result is the square of the distance in the formula of the scaling section. The nearest center to C0002 is center 3, at a squared distance of 4.51. The next nearest is center 1, at 10.17. C0002 is therefore in segment 3, the segment stored in its row of `snap`.

At the end of the fitting, each center is also the mean of its customers. We compare the mean of segment 0's 103 customers with the center of segment 0:

In [ ]:
# --- Look: the mean of segment 0's customers, beside the center of segment 0 ---
pd.DataFrame({"mean of segment 0": X[snap["segment"] == 0].mean(),
              "center 0": centers.iloc[0]}).round(2)

The two columns are equal. With the same pieces, we can compute the inertia by hand. For each segment, we subtract the center from every customer's row, square the differences, and add them up:

In [ ]:
total = 0
for j in range(4):
    members = X[snap["segment"] == j]                         # the scaled rows of segment j
    total = total + ((members - centers.iloc[j]) ** 2).sum().sum()

# --- Look: the inertia computed by hand, and as stored by the model ---
print(round(total, 2), round(km.inertia_, 2))

With `members - centers.iloc[j]`, the center is subtracted from every row of `members`. The first `.sum()` returns the total of each of the six columns, and the second returns the sum of these six totals. Our total, 2302.01, equals the inertia stored in the model's attribute `inertia_`.

The online edition has an interactive explorer here: k-means on the 872 customers and two scaled features, with buttons for the assign and update steps, new random starts, and a list of the final inertia of every run. [Open the interactive explorer](https://pyba.murtaza.cc/assets/demos/kmeans-steps.html).


### Initial centers and `n_init`

As noted above, the grouping reached by k-means depends on the starting centers. To see how much this matters for the six features, we fit k-means ten times with one start each, changing only `random_state`, with which the random choice of the starting centers is fixed:

In [ ]:
rows = []
for seed in range(10):
    m = KMeans(n_clusters=4, n_init=1, random_state=seed).fit(X)
    rows.append([seed, m.inertia_])

# --- Look: the inertia reached from ten different starts ---
pd.DataFrame(rows, columns=["random_state", "inertia"]).round(1)

An inertia of 2302.0 is reached from five of the ten starts, 2396.3 from four, and 2395.8 from one. In the five starts with a higher inertia, the inertia cannot be lowered by a single assign or update step, even though a grouping with a lower inertia exists. With `n_init=10`, k-means is run from ten starts, and the result with the lowest inertia, 2302.0, is kept as the chapter's model, `km`. With `random_state=0`, the same ten starts are used on every run, so if you use the same `random_state`, your segments will match the segments in this book.

If `n_init` is not set, its default, `n_init="auto"`, is used, which means a single start. The starting centers of that start are chosen with a method called **k-means++**. The first center is a customer drawn at random. Each further center is also drawn at random, but customers far from the centers already chosen are more likely to be drawn. In this way, the starting centers are spread out, but the lowest inertia is not guaranteed: in the table above, a single start ended at a higher inertia in five of the ten runs. We therefore set `n_init=10` in every k-means model of this chapter.

## Number of segments

The number of segments, k, is a hyperparameter in the sense of [Chapter 10](https://pyba.murtaza.cc/parts/part-04-predictive/ch-10-classification-2.html): we set it before fitting. Without a label, there is no accuracy to cross-validate, as we did for the tree depth in Chapter 10. We compare values of k with two measures:

- **Inertia.** Inertia falls as k rises, since with more centers each customer is closer to one of them. With 872 centers, every customer would have its own center, and the inertia would be 0. This is the same problem as the training accuracy of the trees in [Chapter 10](https://pyba.murtaza.cc/parts/part-04-predictive/ch-10-classification-2.html#sec-ch10-depth), which rose with every added level: a measure that always improves with a more flexible model cannot choose the setting by itself. We therefore look for the k after which the inertia falls much less with each extra segment. On a chart of inertia against k, that point is a bend, called the **elbow**.
- **Silhouette.** For each customer, we compare the distance to its own segment with the distance to the nearest other segment. The following table lists the definitions.

| Quantity | Definition |
|---|---|
| a | the mean distance from the customer to the other customers in its own segment |
| b | the mean distance from the customer to the customers of the nearest other segment |
| silhouette value | $\dfrac{b - a}{\max(a, b)}$, a number between −1 and 1 |
| silhouette score | the mean of the silhouette values of all customers |

A value near 1 means that the customer is much closer to its own segment than to any other. A value near 0 means that the customer lies between two segments. A negative value means that the customer is closer, on average, to the customers of another segment. The figure shows a and b for one point in a small hypothetical example with three groups.

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
_groups = np.array([0] * 5 + [1] * 5 + [2] * 5)   # the three groups found above
_i = 7                                             # the point whose silhouette is shown
_d = np.sqrt(((_pts - _pts[_i]) ** 2).sum(axis=1)) # its distance to every point
_own = (_groups == _groups[_i]) & (np.arange(len(_pts)) != _i)
_a = _d[_own].mean()
_b, _near = min((_d[_groups == j].mean(), j) for j in range(3) if j != _groups[_i])
_fig = go.Figure()
for _mask, _dash in [(_own, "solid"), (_groups == _near, "dot")]:
    for _p in _pts[_mask]:
        _fig.add_trace(go.Scatter(x=[_pts[_i][0], _p[0]], y=[_pts[_i][1], _p[1]], mode="lines", hoverinfo="skip",
                                  line=dict(color="#555", width=1.2, dash=_dash), showlegend=False))
_fig.add_trace(go.Scatter(x=_pts[:, 0], y=_pts[:, 1], mode="markers", showlegend=False,
                          marker=dict(size=9, color=[_colors[g] for g in _groups], line=dict(width=1, color="white"))))
_fig.add_trace(go.Scatter(x=[_pts[_i][0]], y=[_pts[_i][1]], mode="markers", showlegend=False,
                          marker=dict(size=18, color="rgba(0,0,0,0)", line=dict(width=2.5, color="#333"))))
_fig.add_annotation(x=2.35, y=0.2, xanchor="left", align="left", showarrow=False, font=dict(size=12),
                    text=(f"<b>a</b> = {_a:.2f}<br>mean distance to its own group<br><br>"
                          f"<b>b</b> = {_b:.2f}<br>mean distance to the nearest<br>other group<br><br>"
                          f"<b>silhouette</b> = (b − a) / max(a, b)<br>= ({_b:.2f} − {_a:.2f}) / {_b:.2f} = {(_b - _a) / max(_a, _b):.2f}"))
_fig.update_xaxes(range=[-2.3, 6.2], visible=False)
_fig.update_yaxes(range=[-2.1, 2.1], visible=False, scaleanchor="x")
_fig.update_layout(height=320, margin=dict(t=10, l=10, r=10, b=10), plot_bgcolor="rgba(0,0,0,0)")
_fig

We compute a and b for C0002:

In [ ]:
from sklearn.metrics import silhouette_samples

i = 1                                                   # C0002, in segment 3
dist = ((X - X.iloc[i]) ** 2).sum(axis=1) ** 0.5        # distance to every customer
mean_dist = dist.drop(i).groupby(snap["segment"]).mean()

# --- Look: C0002's mean distance to the customers of each segment ---
mean_dist.round(3)

- **`.sum(axis=1)`** returns one total per row, the sum across the columns of that row. Without `axis=1`, `.sum()` returns one total per column, as in the inertia cell above. Here each row's total is the squared distance from C0002 to that customer.
- **`** 0.5`**: with this operation, a number is raised to the power one half, which is its square root. In this way, the squared distances are converted to distances.
- **`.drop(i)`** returns the column without the row labeled `i`, here C0002's distance to itself, which is 0.
- **`dist.groupby(snap["segment"])`**: with this call, the distances are grouped by the segment column of `snap`. Both have one row per customer, and rows with the same label are matched.

Because C0002 is in segment 3, a is its mean distance to the other 210 customers of segment 3: 2.542. Of the other three segments, segment 1 is the nearest on average, with a mean distance of 3.499. Therefore b = 3.499. The silhouette value follows from the definition:

In [ ]:
a = mean_dist[3]
b = mean_dist.drop(3).min()                        # the nearest other segment
print(f"a = {a:.3f}, b = {b:.3f}, silhouette = {(b - a) / max(a, b):.3f}")
print(f"silhouette_samples: {silhouette_samples(X, snap['segment'])[i]:.3f}")

C0002's silhouette value is

$$\frac{3.499 - 2.542}{\max(2.542,\ 3.499)} = \frac{0.957}{3.499} = 0.274$$

`silhouette_samples`, from scikit-learn's `metrics` module, returns the silhouette value of every customer as an array. Its value for C0002 is the same. `silhouette_score` returns the mean of these values. We now compute the inertia and the silhouette score for k from 2 to 7:

In [ ]:
from sklearn.metrics import silhouette_score

rows = []
for k in range(2, 8):
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    rows.append([k, m.inertia_, silhouette_score(X, m.labels_)])

kdf = pd.DataFrame(rows, columns=["k", "inertia", "silhouette"])

# --- Look: inertia and silhouette score for each k ---
kdf.round(3)

`range(2, 8)` returns the numbers 2 to 7: the stop value, 8, is excluded. After `.fit`, the attribute `labels_` holds each customer's group number, the same numbers that `fit_predict` returns.

In [ ]:
fig = px.line(kdf.melt(id_vars="k", var_name="measure"), x="k", y="value",
              facet_row="measure", markers=True, height=420)
fig.update_yaxes(matches=None)

# Label each drop in inertia, between neighboring values of k, in the inertia panel
for k, before, after in zip(kdf["k"][1:], kdf["inertia"][:-1], kdf["inertia"][1:]):
    fig.add_annotation(x=k - 0.5, y=(before + after) / 2, text=f"−{before - after:,.0f}",
                       showarrow=False, xanchor="left", yanchor="bottom", row=2, col=1)

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
# panel names in sentence case, and the two values the text discusses marked on the chart
# (the facet labels move to the y-axis titles on the left; the x-axis title goes on the bottom panel only)
fig.for_each_annotation(lambda a: a.update(text="") if a.text.startswith("measure=") else None)
fig.update_yaxes(title_text="Inertia", row=2, col=1)
fig.update_yaxes(title_text="Silhouette score", row=1, col=1)
fig.update_xaxes(title_text="")
fig.update_xaxes(title_text="Number of segments, k", row=1, col=1)
_best = kdf.loc[kdf["silhouette"].idxmax()]
fig.add_annotation(x=_best["k"], y=_best["silhouette"], ax=40, ay=-4, xanchor="left", showarrow=True, arrowhead=2,
                   text=f"Highest silhouette score: {_best['silhouette']:.3f} at k = {_best['k']:.0f}", row=1, col=1)
fig.add_vline(x=4, line=dict(color="#cf222e", width=1.5, dash="dot"))
fig.add_annotation(x=4, y=1, yref="paper", yanchor="bottom", showarrow=False, text="<b>Chosen: k = 4</b>",
                   font=dict(color="#cf222e"))
fig.update_layout(margin=dict(t=40))
fig

To draw both measures, we reshape `kdf` with `melt`, as in [Chapter 10](https://pyba.murtaza.cc/parts/part-04-predictive/ch-10-classification-2.html), and draw one panel per measure with `facet_row`, as in [Chapter 11](https://pyba.murtaza.cc/parts/part-04-predictive/ch-11-forecasting.html). By default, every panel of a faceted Plotly chart has the same y-axis range. Because inertia is in the thousands and the silhouette score is below 1, the silhouette curve would be flat on a shared range. We therefore give each panel its own range with `fig.update_yaxes(matches=None)`. With `update_yaxes`, a setting is changed on both y-axes of the figure, and with `matches=None`, the two y-axes no longer share one range.

We cannot read a single k from either measure. The inertia falls by less at each step (1,019, 493, 278, 193, and 141), and the curve has no single sharp bend. The silhouette score is highest at k = 3. With the two measures, we narrow the choice to three or four segments. To see how the groups change from three segments to four, we compare the three-segment and four-segment groupings with a crosstab, as in [Chapter 5](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-05-relationships.html):

In [ ]:
km3 = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)

# --- Look: where the customers of each k = 4 segment go at k = 3 ---
pd.crosstab(km3.labels_, snap["segment"], rownames=["k = 3"], colnames=["k = 4"])

With `rownames` and `colnames`, the two axes of the crosstab are labeled. Segments 0 and 3 are almost the same at both values of k. The difference lies in the large middle group: at k = 3, segment 2 (205 customers) and 337 of the 353 customers of segment 1 form a single group, and at k = 4 they are separated. We need this separation for the marketing plan. In the profiles below, the customers of segment 2 place about 33 orders a year and those of segment 1 about 13, and in the plan at the end of the chapter, the two segments receive different treatments. We therefore choose k = 4, and in the report to the marketing team, we include the figure together with this reasoning.

## Segment profiles

From a segment number alone, the marketing team learns nothing about the customers in it. For each segment we need a profile, which is a description in the original units, and a name based on the profile. We start by converting the segment centers back to the original units. With the scaler's `inverse_transform`, the scaling is reversed: each value is multiplied by the feature's standard deviation, and the feature's mean is added back:

$$\text{value} = \text{scaled value} \times \text{standard deviation} + \text{mean}$$

In [ ]:
# --- Look: the four centers in the original units ---
original = scaler.inverse_transform(km.cluster_centers_)
pd.DataFrame(original, columns=features).round(2)

In the original units, the center of segment 0 has 8.05 orders a year and an average order of \$2,198.24. The center is the mean of the segment's customers. In scaling, each feature is only shifted and stretched. The center in the original units is therefore also the mean of the customers' original values. We compute these means directly with `groupby`, and add twelve-month revenue, which was not used in the clustering, as context:

In [ ]:
profile = snap.groupby("segment")[features + ["revenue_12m"]].mean().round(2)
profile["customers"] = snap["segment"].value_counts()      # matched by segment number
profile

The first six columns contain the same numbers as the centers in the previous cell. We count the business types in each segment with a crosstab. With `normalize="index"`, each row is divided by its total, so that the shares in each row sum to 1:

In [ ]:
pd.crosstab(snap["segment"], snap["business_type"], normalize="index").round(2)

From the two tables, we name the segments. Each name is a summary of the segment means in the profile:

| Segment | Name | Customers | Profile (segment means) |
|---|---|---|---|
| 0 | Seasonal institutions | 103 | 8.05 orders a year, \$2,198 per order, 37% of orders in August or January (20% across all customers); 81% schools and 19% nonprofits; \$18,325 revenue |
| 1 | Mid-range accounts | 353 | every feature within half a standard deviation of the overall mean (12.85 orders, \$286 per order); every business type present; \$3,727 revenue |
| 2 | Frequent buyers | 205 | 33.10 orders a year, \$460 per order, last order 1.62 weeks before the cutoff; 75% restaurants and 22% retail; \$15,454 revenue |
| 3 | Small online buyers | 211 | 10.88 orders a year, \$87 per order, 94% of orders online, average discount 4%; \$950 revenue |

The average discount of the small online buyers, 4%, is the lowest of the four segments. In [Chapter 5](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-05-relationships.html) we saw that reps authorize the deepest discounts. These customers place 94% of their orders online. At most 6% of their orders can therefore include a discount authorized by a rep. Customers inside a segment vary around the segment means. Many lie close to the boundary with another segment, as we see from the silhouette values in the evaluation.

We attach the names to the snapshot with a dictionary from segment number to name, as in [Chapter 2](https://pyba.murtaza.cc/parts/part-01-foundations/ch-02-python-essentials.html). With `.map`, each value of the column is looked up in the dictionary, and the matching name is returned:

In [ ]:
names = {0: "Seasonal institutions", 1: "Mid-range accounts",
         2: "Frequent buyers", 3: "Small online buyers"}
snap["segment_name"] = snap["segment"].map(names)

# --- Look: the first three customers with their segment names ---
snap[["customer_id", "segment", "segment_name"]].head(3)

In [ ]:
fig = px.scatter(snap, x="orders_12m", y="avg_order_value", color="segment_name",
                 log_y=True, opacity=0.55,
                 labels={"orders_12m": "Orders (12 months)",
                         "avg_order_value": "Average order value ($, log)",
                         "segment_name": "Segment"})

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
# each segment named on the chart in its own color, in an empty area beside its customers
_col = {t.name: t.marker.color for t in fig.data}
_pos = {"Seasonal institutions": (17, 3000), "Frequent buyers": (52, 900), "Mid-range accounts": (30, 220),
        "Small online buyers": (30, 75)}
for _name, (_x, _y) in _pos.items():
    fig.add_annotation(x=_x, y=__import__("math").log10(_y), text=f"<b>{_name}</b>", showarrow=False,
                       xanchor="left", font=dict(size=13, color=_col[_name]))
fig.add_annotation(x=17.5, y=__import__("math").log10(240), ax=29.5, ay=__import__("math").log10(220), axref="x",
                   ayref="y", showarrow=True, arrowhead=2, arrowcolor=_col["Mid-range accounts"], text="")
_ticks = [50, 100, 200, 500, 1000, 2000, 5000]
fig.update_yaxes(tickvals=_ticks, ticktext=[f"${t:,}" for t in _ticks])
fig.update_layout(showlegend=False)
fig

## Evaluation against business types
With the segments named, we evaluate them with the three checks listed in Chapter 12. The baseline is the business-type grouping, the existing grouping with which the segments are compared, in the role of the baselines of Chapters 9 to 11.

|  |  |
|---|---|
| **Metrics** | the silhouette score on the six scaled features; the adjusted Rand index (ARI) between the chapter's segments and refitted segments; the churn rate and revenue of each group |
| **Test** | a refit with a different `random_state`; refits on ten random subsamples of 80% of the customers; churn in the 26 weeks after the cutoff, a column not used in the clustering |
| **Baseline** | the six business types, the existing grouping of the customers |

The first check is separation, which measures how distinct the segments are from one another. We compute the silhouette score of the segments and of the business types on the same scaled features:

In [ ]:
print(f"silhouette, four segments:      {silhouette_score(X, snap['segment']):.3f}")
print(f"silhouette, six business types: {silhouette_score(X, snap['business_type']):.3f}")

The four segments have a silhouette score of 0.296, a moderate value. The business types have a score of −0.008. On these six features, a customer is on average about as close to customers of other business types as to customers of its own. Because k-means was fitted to separate customers on exactly these features, this comparison is weighted toward k-means. From this comparison, we can conclude only that customers of the same business type do not buy alike on these six features.

The second check is stability, which measures whether the same segments appear again when the clustering is repeated. A stable grouping is reproduced by a refit from different starting centers, and by a refit on a different sample of customers. The **adjusted Rand index (ARI)** is a measure of agreement between two groupings of the same customers. To compute it, we count the pairs of customers that are placed together in both groupings, or apart in both, and adjust the count for the agreement expected by chance. The ARI is 1 when the two groupings are identical, whatever numbers are assigned to the groups. The ARI is near 0 when the groupings agree no more than chance would, and it can be slightly negative.

In [ ]:
from sklearn.metrics import adjusted_rand_score

km_b = KMeans(n_clusters=4, n_init=10, random_state=99).fit(X)
print(f"refit with random_state=99: ARI {adjusted_rand_score(snap['segment'], km_b.labels_):.2f}")

aris = []
for r in range(10):
    part = X.sample(frac=0.8, random_state=r)              # 80% of the customers
    m = KMeans(n_clusters=4, n_init=10, random_state=0).fit(part)
    aris.append(adjusted_rand_score(snap["segment"], m.predict(X)))

print(f"refits on ten 80% subsamples: ARI from {min(aris):.3f} to {max(aris):.3f}")

`adjusted_rand_score` is imported from scikit-learn's `metrics` module. It returns the ARI of two groupings. With `X.sample(frac=0.8, random_state=r)`, 80% of the rows of `X` are drawn at random, without replacement. The model is fitted on those rows. With `m.predict(X)`, every one of the 872 customers is then assigned to the nearest of the model's four centers, so that the result can be compared with `snap["segment"]`.

With another `random_state`, the segments are reproduced exactly (ARI 1.00). The segments from the ten subsamples agree with the chapter's segments at an ARI between 0.975 and 1.000. From each subsample, 174 customers are left out, and the four segments are found again each time. The segments therefore depend on the customers' purchasing behavior and only slightly on the random starts or on which customers are in the sample.

From the per-customer silhouette values, we can see where the segments are less distinct. We count the customers with a negative value in each segment:

In [ ]:
sil = silhouette_samples(X, snap["segment"])        # one value per customer

# --- Look: customers with a negative silhouette value, by segment ---
snap.loc[sil < 0, "segment_name"].value_counts()

With `snap.loc[sil < 0, "segment_name"]`, we select the segment names of the customers whose silhouette value is below zero. Of the 872 customers, 42 have a negative value. Of these 42, 38 are mid-range accounts. The center of the mid-range segment lies within half a standard deviation of the overall mean on every feature. Customers at the edges of this segment can therefore be close to another segment as well.

The online edition has a per-customer silhouette explorer here: every customer's silhouette value, sorted within its segment, with each customer's mean distance to the four segments and the resulting a, b, and silhouette value. [Open the interactive explorer](https://pyba.murtaza.cc/assets/demos/silhouette-clusters.html).


The third check is usefulness for a decision. A grouping is useful to the marketing and account teams if its groups differ on outcomes the business acts on. We compare the churn rate and the revenue of the segments with those of the business types. The `churned` column is the Chapter 9 label: 1 if the customer placed no order in the 26 weeks after the cutoff. It was not used in the clustering.

In [ ]:
by_segment = snap.groupby("segment_name")[["churned", "revenue_12m"]].mean()
by_type = snap.groupby("business_type")[["churned", "revenue_12m"]].mean()

# --- Look: churn rate and mean revenue, by segment and by business type ---
print(by_segment.round(3), "\n")
print(by_type.round(3))

The churn rate of the segments ranges from 4.9% among the frequent buyers to 39.3% among the small online buyers. The churn rate of the business types ranges only from 15.1% (schools) to 27.2% (retail). Mean revenue ranges from \$950 to \$18,325 across the segments, and from \$1,610 to \$15,956 across the business types. The spread in churn, the highest rate minus the lowest, is 39.3 − 4.9 = 34.4 percentage points across the segments and 27.2 − 15.1 = 12.1 points across the business types. A retention plan can therefore be targeted more precisely by segment. If the segments did not differ on such outcomes, there would be no reason to treat them differently, however well separated and stable they were.

## Segment dashboard

To make the segments available to the marketing team, we add them to the Streamlit dashboard that we built in [Chapter 6](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-06-dashboards.html). First we export the snapshot with its segment columns:

In [ ]:
export_path = DATA_DIR / "cache" / "pw_segments.csv"
export_path.parent.mkdir(parents=True, exist_ok=True)    # create the cache folder if needed
snap.to_csv(export_path, index=False)

# --- Look: the size of the exported table ---
snap.shape

The file has one row per customer, 872 in all, with the new columns `segment` and `segment_name`. `.parent` is the folder that contains the file. With `mkdir(parents=True, exist_ok=True)`, the folder is created if it does not exist yet. With `index=False`, the row labels are not written to the file.

In `dashboards/monday_dashboard.py`, we then add a segment filter in two places. In `load_lines`, after the merge with the customer table, the segment names are merged onto the order lines. The 28 customers who placed no order in the year before the cutoff are not in the snapshot. Their order lines therefore have no segment name. To keep the missing value out of the list of choices, we remove it with `.dropna()`. In the filter section, the new control and the mask follow the region filter:

```python
# in load_lines(), after the merge with the customer table
segments = pd.read_csv(DATA_DIR / "cache" / "pw_segments.csv")
lines = lines.merge(segments[["customer_id", "segment_name"]],
                    on="customer_id", how="left", validate="many_to_one")

# in the filter section, after the region filter
chosen = st.sidebar.multiselect("Segment", sorted(lines["segment_name"].dropna().unique()))
if chosen:
    view = view[view["segment_name"].isin(chosen)]
```

Because every KPI and chart is built from `view`, each can now be viewed for one segment or several. Before you start the dashboard, run this chapter's notebook through the export cell, so that `pw_segments.csv` exists.

## The decision this informs

We give the four segments four different treatments, each based on the profile and the evaluation numbers:

| Segment | Treatment | Basis in the numbers |
|---|---|---|
| Seasonal institutions | a named account manager, with outreach in July and December, before the August and January orders | 37% of their orders fall in August or January; \$18,325 revenue per customer |
| Frequent buyers | first priority in delivery scheduling | 33 orders a year, about one every 11 days; \$15,454 revenue per customer |
| Mid-range accounts | the main audience of the Chapter 9 retention list | the largest segment, 353 customers; churn 23.8%; \$3,727 revenue per customer |
| Small online buyers | the free-shipping rollout of Chapter 7, tracked by segment | 94% of orders online; churn 39.3%; \$950 revenue per customer |

In the figure, the four segments are drawn with their treatments by churn rate and mean revenue, together with the six business types. The segments lie farther apart than the business types on both measures.

In [ ]:
#@title Run this cell to draw the figure (plotting code only)
import plotly.graph_objects as _go

# segments (colored, sized by customers) and business types (gray) on churn rate and mean revenue
_counts = snap["segment_name"].value_counts()
_treat = {"Seasonal institutions": "Named account manager", "Frequent buyers": "First priority in delivery",
          "Mid-range accounts": "Retention list", "Small online buyers": "Free-shipping rollout"}
_f = _go.Figure()
_f.add_trace(_go.Scatter(x=by_type["churned"] * 100, y=by_type["revenue_12m"], mode="markers+text",
                         marker=dict(size=10, color="rgba(0,0,0,0)", line=dict(width=1.5, color="#8c959f")),
                         text=[t.capitalize() for t in by_type.index], textposition="middle right",
                         textfont=dict(size=11, color="#6e7781"), name="Business type"))
_f.add_trace(_go.Scatter(x=by_segment["churned"] * 100, y=by_segment["revenue_12m"], mode="markers+text",
                         marker=dict(size=[(_counts[s] ** 0.5) * 2 for s in by_segment.index], color="#0969da",
                                     opacity=0.75),
                         text=[f"<b>{s}</b><br>{_treat[s]}" for s in by_segment.index], textposition=["top center" if s != "Mid-range accounts" else "middle left" for s in by_segment.index],
                         textfont=dict(size=12), name="Segment (size: customers)"))
_lo, _hi = round(by_segment["churned"].min() * 100, 1), round(by_segment["churned"].max() * 100, 1)
_tl, _th = round(by_type["churned"].min() * 100, 1), round(by_type["churned"].max() * 100, 1)
for _a, _b, _y, _lab in [(_lo, _hi, -2500, "segments"), (_tl, _th, -4500, "business types")]:
    _f.add_annotation(x=_b, y=_y, ax=_a, ay=_y, xref="x", yref="y", axref="x", ayref="y", showarrow=True,
                      arrowhead=2, startarrowhead=2, arrowside="end+start", arrowwidth=1.5, text="")
    _f.add_annotation(x=(_a + _b) / 2, y=_y, yshift=10, showarrow=False, font=dict(size=11),
                      text=f"Spread in churn, {_lab}: {_b - _a:.1f} points")
_f.update_xaxes(title_text="Churn rate in the 26 weeks after the cutoff (%)", range=[0, 45])
_f.update_yaxes(title_text="Mean revenue in the 12 months before the cutoff", tickprefix="$", range=[-6500, 22000],
                 tickvals=[0, 5000, 10000, 15000, 20000],
                 ticktext=["$0", "$5k", "$10k", "$15k", "$20k"])
_f.update_layout(height=460, legend=dict(orientation="h", y=-0.2, x=0), margin=dict(t=20))
_f

For the small online buyers, we use the retention list with care. In the Chapter 9 cost figures, a loss of \$600 in margin per churner is assumed, but the margin from a customer with \$950 of revenue a year is probably smaller. With a smaller loss, the break-even threshold for an offer (Chapter 9) is higher for this segment.

One limitation of the plan is that the segments are based on customer behavior up to a single cutoff date. Because customers move between segments as their behavior changes, we refit the segments, and repeat the stability check, with every new snapshot.

## Exercises


### Build lab

The chapter's segments are based on six behavioral features. In this lab, you will segment the customers on *value*, with the scaled `revenue_12m`, `orders_12m`, and `tenure_months` only. Use k = 4, so that the value segments can be compared with the behavioral segments. For reference, report the inertia and the silhouette score for k from 2 to 7. Profile the value segments in the original units, and state for each whether its mean revenue, orders, and tenure are above or below the overall means. Compare the two groupings in a crosstab. For each behavioral segment, report the value segment that contains most of its customers, and that share.

### Evaluate lab

Evaluate your value segments with the chapter's three checks. Compute their silhouette score on the three scaled value features, beside the silhouette score of the business types on the same three features. Compute the ARI between your segments and refits on ten 80% subsamples. Report the churn rate of each value segment beside the chapter's segments. Compute the spread in churn rates of each grouping, the highest rate minus the lowest, in percentage points. As in the chapter's usefulness check, we give the retention team the grouping with the larger spread. Name that grouping.

A starter notebook for this lab is provided. In it, the task is restated, the objects from this chapter that are needed for the lab are reproduced, and the cells you complete are marked. [**Open ch-12-lab-starter.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-12-lab-starter.ipynb), or download it from the course page in Blackboard. In Colab, a notebook from GitHub is opened read-only: click **Copy to Drive** in the toolbar before you edit anything, and work in that copy.
